# MedGS4D: endpoint-only respiratory CT interpolation

MedGS represents a CT volume with learnable Gaussian primitives and reconstructs its slices with a differentiable renderer. MedGS4D adds a time-conditioned network to a frozen canonical MedGS representation: instead of fitting a separate volume at every respiratory phase, it moves the same primitives through time.

The task is to reconstruct **10%, 20%, 30%, and 40%** from the observed **0% and 50%** scans. The six variants below test which temporal assumptions help when intermediate scans are unavailable during fitting. Each case is fitted independently; there is no population-level pretraining.

The benchmark targets comparison with **UVI-Net (CVPR 2024)** on its original 4D-Lung preprocessing and test series. Architecture and consistency extensions are hypotheses; their benefit is determined by the final evaluation.


## 1. Experiment configuration

This experiment fits 14 canonical models and 84 temporal models (six methods per series), then reconstructs and evaluates 336 volumes. Each canonical model is trained for 15,000 iterations and each temporal model for 7,000 iterations, with seed 42.

Set the data and repository paths on the machine running the kernel. Running the configuration cell creates a new experiment ID; run the notebook from top to bottom to compute all results from scratch.


In [ ]:
from pathlib import Path

import hashlib
import json
import os
import platform
import subprocess
import sys
from datetime import datetime, timezone

STACK_ROOT = Path("/home/jovyan/shared/mtm_medgs_stack")
DATA_ROOT = STACK_ROOT / "data" / "uvi_net" / "4D-Lung-Preprocessed"
REPO_ROOT = STACK_ROOT / "repo" / "medgs4d"
EXPERIMENT_ID = "medgs4d_replication_" + datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f")
ROOT = STACK_ROOT / "experiments" / EXPERIMENT_ID
GPU_INDEX = "1"  # Index among GPUs visible to this kernel; workers use cuda:0.
SEED = 42
CANONICAL_ITERATIONS = 15_000
TEMPORAL_ITERATIONS = 7_000
PHASES = [10, 20, 30, 40]
OFFICIAL_TEST_CASES = [f"118_{i}" for i in range(6)] + [f"119_{i}" for i in range(8)]
TEST_CASES = OFFICIAL_TEST_CASES.copy()
METHODS = ["baseline", "linear_pseudo", "cycle_v2", "trajectory", "consistent_cycle", "trajectory_cycle"]
LABELS = dict(zip(METHODS, ["Baseline", "Linear pseudo", "Cycle-v2", "Endpoint trajectory", "Consistent cycle",
                            "Trajectory + cycle"]))
NEW_OPTIONS = {
    "endpoint_warmup": 1000, "transport_pretrain": 500, "joint_ramp": 1000,
    "endpoint_image_weight": 0.05, "alignment_weight": 0.01,
    "roundtrip_weight": 0.01, "path_weight": 0.005,
    "correction_weight": 1e-3, "curvature_weight": 1e-5,
    "curvature_step": 0.025, "geometry_gaussians": 8192,
}
QUALITATIVE_CASES = list(dict.fromkeys([TEST_CASES[0], TEST_CASES[len(TEST_CASES) // 2], TEST_CASES[-1]]))
QUALITATIVE_SLICE = 64
ERROR_VMAX = 0.2
print("Experiment:", ROOT)
print("Budget:", len(TEST_CASES), "canonical +", len(TEST_CASES) * len(METHODS), "temporal runs")


### Environment

The run records library versions, GPU details and the source revision. CUDA rendering can introduce small differences between repeated runs even with the same seed.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nibabel as nib
import torch
from IPython.display import display
import benchmark.metrics as benchmark_metrics
from benchmark.environment import validate_environment

WORKER_GPU = validate_environment(REPO_ROOT, GPU_INDEX, benchmark_metrics)
plt.style.use("default")
ROOT.mkdir(parents=True, exist_ok=False)
for folder in ["config", "environment", "prepared", "canonical", "methods", "predictions",
               "metrics", "tables", "figures", "logs"]:
    (ROOT / folder).mkdir()


def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def write_json(path, value):
    Path(path).write_text(json.dumps(value, indent=2, sort_keys=True) + "\n")


runtime = {
    "python": platform.python_version(), "interpreter": sys.executable,
    "torch": torch.__version__, "numpy": np.__version__, "cuda": torch.version.cuda,
    "cudnn": torch.backends.cudnn.version(), "gpu_index": GPU_INDEX,
    "gpu": torch.cuda.get_device_name(int(GPU_INDEX)),
}
write_json(ROOT / "environment/runtime.json", runtime)
display(pd.DataFrame(runtime.items(), columns=["Setting", "Value"]))


In [ ]:
from benchmark.provenance import record_sources

source_record = record_sources(REPO_ROOT, ROOT / "config/source_manifest.json")
print("Source commit:", source_record["commit"])


## 2. UVI-Net benchmark split and information available to the model

The goal is to add our results to the same benchmark comparison as UVI-Net. We therefore retain its original CVPR preprocessing and folder split: sort 82 series, use the first 68 as the official training split and the last 14 as the test split. These test series are `118_0`–`118_5` and `119_0`–`119_7`, from **two patients**. They are not 14 independent patients.

MedGS4D fits an individual representation to the two observed endpoints of each test series. This is **test-time, instance-specific fitting**, whereas UVI-Net learns across training data. The same hidden targets make interpolation scores comparable, but fitting time and the learning setting must accompany the table.

Only `frame0` and `frame5` enter prepared training data. Intermediate ground truth is read after all six models on all 14 series have finished; it is not used for tuning, checkpoint choice or early stopping. The fixed iteration budget is a controlled starting point, not an assumed optimum. Any later tuning should use development series from the official first 68 and a new experiment ID.

Case `118_0` was inspected during earlier development. It remains in the main table to preserve the official split; an additional table excludes it. This supplementary analysis does not make the original test set wholly untouched.

Sources: [UVI-Net repository and original data release](https://github.com/jungeun122333/UVI-Net), [evaluation protocol](https://github.com/jungeun122333/UVI-Net/blob/main/evaluation.py), [metric definitions](https://github.com/jungeun122333/UVI-Net/blob/main/losses.py).


In [ ]:
assert DATA_ROOT.is_dir(), f"Brak katalogu danych: {DATA_ROOT}"
print("Dane:", DATA_ROOT)

folders = sorted(p.name for p in DATA_ROOT.iterdir() if p.is_dir())
assert len(folders) == 82, f"Expected 82 series, found {len(folders)}."
assert folders[68:] == OFFICIAL_TEST_CASES, folders[68:]
assert set(TEST_CASES) <= set(OFFICIAL_TEST_CASES)
for case in TEST_CASES:
    for frame in range(6):
        assert (DATA_ROOT / case / f"ct_{case}_frame{frame}.nii.gz").is_file()
SPLIT = {"preprocessing": "UVI-Net CVPR 2024 original 4D-Lung-Preprocessed",
         "train": folders[:68], "test": OFFICIAL_TEST_CASES,
         "selected_cases": TEST_CASES,
         "observed_phases": [0, 50], "evaluation_phases": PHASES,
         "previously_inspected_test_cases": ["118_0"]}
write_json(ROOT / "config/data_split.json", SPLIT)
display(pd.DataFrame({"Case": TEST_CASES, "Patient": [c.split("_")[0] for c in TEST_CASES]}))


## 3. Six temporal models

Let $G_0$ be the frozen canonical Gaussian set, $\mathcal R(G)$ its slice renderer, and $u=\text{phase}/50\in[0,1]$ the local time between the observed scans. A state $q(u)$ contains Gaussian geometry; $d(u)$ denotes its displacement in canonical-normalized $(x,z,m)$ coordinates. Here $m$ is the bounded MedGS depth parameter, represented internally through a logit. The original slice-plane coordinate $y$ remains fixed.

Every temporal model minimizes endpoint reconstruction
\[
L_{\rm image}=2\|\mathcal R(G(1))-I_{50}\|_1+0.25(1-\operatorname{SSIM}_{2D}),
\]
plus the same displacement penalties ($10^{-4}$ magnitude, $10^{-3}$ endpoint-to-canonical smoothness). These are training losses; the reported SSIM is a separate full-volume 3D metric.

| ID | Primary trajectory | Additional training signal | Inference |
|:--|:--|:--|:--|
| baseline | Original phase MLP | Endpoint image | Primary field |
| linear_pseudo | Original phase MLP | Linear geometric pseudo-target | Primary field |
| cycle_v2 | Original phase MLP | Existing pairwise endpoint reconstruction | Primary field |
| trajectory | Endpoint + residual heads | Small correction and curvature priors | Primary field |
| consistent_cycle | Original phase MLP | Alignment, round trip, path consistency, endpoint reconstruction | Primary field |
| trajectory_cycle | Endpoint + residual heads | Same consistency losses and trajectory priors | Primary field |

The auxiliary pairwise network is used only during fitting. New variants retain its current architecture to isolate the effects of the objective and primary trajectory; they do not simultaneously change every network component.


### Baseline and linear pseudo-supervision

The baseline predicts $\Delta(u)=f_\theta(g,u/2)-f_\theta(g,0)$. Subtraction anchors the canonical phase exactly. Fitting the endpoint tells the model where to finish, but does not uniquely specify its path through the middle.

Linear pseudo-supervision adds
\[
L_{\rm pseudo}=\|d(u)-u\,\operatorname{sg}[d(1)]\|_2^2,\qquad \lambda_{\rm pseudo}=0.1.
\]
`sg` stops gradients through the target. In plain terms: each Gaussian is encouraged to cover the corresponding fraction of its endpoint displacement. This is a useful straight-path assumption, not an intermediate CT label. The original implementation samples an intermediate time each iteration.


### Cycle-v2: pairwise endpoint reconstruction

A second network $T_\phi(q,s,t)$ moves a Gaussian state from time $s$ to time $t$. “Pairwise” refers to a **pair of time points**. Its displacement is multiplied by $t-s$, so transport from a time to itself is the identity.

Cycle-v2 uses a predicted interior state and linearly extrapolated virtual states before and after the observed interval. It transports these states back to the two observed endpoints, blends endpoint candidates and compares their rendered images with $I_0$ and $I_{50}$. Agreement and transport penalties accompany reconstruction, with weights **0.05 / 0.01 / 0.001**, warm-up 1,000 and ramp 1,000 iterations.

Intuitively: an invented middle state should allow the auxiliary network to recover the scans we actually know. This is an auxiliary reconstruction task; it does not by itself impose an explicit return-to-start cycle or consistency of different transport paths.


### Endpoint–residual trajectory

The primary network has a shared spatial backbone and two heads:
\[
\Delta(u)=u\,A_\theta(g)+u(1-u)B_\theta(g,u).
\]
$A$ learns the endpoint displacement; $B$ can bend the trajectory between endpoints. The factor $u(1-u)$ makes the correction disappear exactly at both ends. Thus $\Delta(0)=0$ and $\Delta(1)=A$ regardless of $B$.

In plain terms: first learn where each primitive finishes, then allow a controlled deviation in the middle. Because endpoint images alone do not identify that deviation, this variant adds weak priors
\[
L_{\rm corr}=\|d(u)-u d(1)\|^2,\qquad
L_{\rm curv}=\left\|\frac{d(u+h)-2d(u)+d(u-h)}{h^2}\right\|^2.
\]
They prefer a small, smooth correction; they do not recover an observed nonlinear motion. The normalized $m$ displacement uses sigmoid-bounded geometry, so linear logit motion need not be linear in $m$. This variant is the control for the changed primary architecture.


### Transport consistency

This variant retains the baseline field and gives the auxiliary transport a more specific task. For model states $q_0,q_1,q_u,q_v$, it adds:

- **Alignment:** transport from either endpoint should meet the primary state at the same intermediate time, $T(q_s,s,u)\approx q_u$.
- **Round trip:** moving away and back should recover the starting geometry, $T(T(q_s,s,u),u,s)\approx q_s$, for $s\in\{0,1\}$.
- **Path consistency:** an indirect route should agree with a direct route, $T(T(q_s,s,u),u,v)\approx T(q_s,s,v)$.
- **Endpoint image:** transport the predicted middle state to an observed endpoint and reconstruct its real CT slice; alternate the two endpoints between iterations.

In plain terms: the two networks must describe compatible motion, a return trip should not drift, and a detour should not change the destination. These are geometric self-consistency assumptions, not hidden labels or proof of physiological correctness.

Training starts with 1,000 endpoint iterations, then 500 iterations teaching transport against detached primary states. Joint coupling ramps over the next 1,000 iterations. Alignment uses two opposite stop-gradient directions; round-trip targets and direct-path targets are detached. Geometry losses use a fixed subset of up to 8,192 Gaussians, while endpoint image losses render the full state.


### Endpoint–residual trajectory with transport consistency

The final variant combines the endpoint/residual architecture with the losses above. The pairwise network provides an additional constraint on the otherwise underdetermined residual trajectory. The small-correction and curvature priors remain active.

The comparison answers two separate questions: does the new primary trajectory help on its own, and does consistency help with either primary architecture? All three new variants are experimental hypotheses. Their weights below are fixed starting values rather than values selected on the 14 test series. Equal iteration counts do not imply equal computation, so fitting time and memory are reported as well.


In [ ]:
display(pd.DataFrame([{"Parameter": k, "Value": v} for k, v in NEW_OPTIONS.items()]))
print("Training budget per case:", CANONICAL_ITERATIONS, "canonical;", TEMPORAL_ITERATIONS, "per temporal method")


## 4. Training and prediction

Temporal models are defined in `models/deformation.py`, `models/trajectory.py` and `models/transport.py`; optimization is in `training/`. The canonical model and temporal models share the MedGS Gaussian representation and renderer.

`benchmark.worker` runs preparation, fitting and prediction in separate processes, releasing GPU memory between runs.


### Worker: endpoint preparation, fitting and prediction

Preparation only transposes the supplied normalized volumes from `(X,Y,Z)` to `(Z,Y,X)` and stores float32 arrays. There is no additional registration, denoising or intensity normalization. Canonical fitting reads 8-bit PNG slices. Temporal fitting also quantizes target slices to 8 bits and converts them to float tensors in [0,1]. Full-volume evaluation uses the supplied float32 ground truth.

Workers select the final numbered checkpoint explicitly. Rendering loads the correct primary architecture for each variant; it does not use the auxiliary network. RGB predictions are clipped to `[0,1]` and averaged to grayscale, matching the existing MedGS4D rendering convention. The fraction of clipped RGB values is recorded.


In [ ]:
E = {
    "root": str(ROOT), "repo_root": str(REPO_ROOT.resolve()), "data_root": str(DATA_ROOT), "seed": SEED,
    "gpu_index": GPU_INDEX, "canonical_iterations": CANONICAL_ITERATIONS,
    "temporal_iterations": TEMPORAL_ITERATIONS, "test_cases": TEST_CASES,
    "methods": METHODS, "evaluation_phases": PHASES, "new_options": NEW_OPTIONS,
    "source_manifest_sha256": sha256_file(ROOT / "config/source_manifest.json"),
    "method_source_sha256": sha256_file(REPO_ROOT / "training/methods.py"),
    "worker_source_sha256": sha256_file(REPO_ROOT / "benchmark/worker.py"),
    "qualitative_cases": QUALITATIVE_CASES, "qualitative_slice": QUALITATIVE_SLICE,
    "error_vmax": ERROR_VMAX,
}
CONFIG_PATH = ROOT / "config/experiment.json"
write_json(CONFIG_PATH, E)

WORKER_ENV = os.environ.copy()
WORKER_ENV["CUDA_VISIBLE_DEVICES"] = WORKER_GPU
WORKER_ENV["PYTHONHASHSEED"] = str(SEED)
WORKER_ENV["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
WORKER_ENV.pop("PYTHONPATH", None)
WORKER_ENV["PATH"] = str(Path(sys.executable).parent) + os.pathsep + WORKER_ENV.get("PATH", "")


def run_worker(mode, case=None, method=None):
    command = [sys.executable, "-u", "-m", "benchmark.worker", str(CONFIG_PATH), mode]
    if case is not None:
        command += ["--case", case]
    if method is not None:
        command += ["--method", method]
    log = ROOT / "logs" / ("_".join(str(x) for x in [mode, case, method] if x is not None) + ".log")
    print("Starting:", mode, case or "", method or "", flush=True)
    with log.open("a") as stream:
        stream.write("\n" + datetime.now(timezone.utc).isoformat() + "\n")
        process = subprocess.Popen(command, cwd=REPO_ROOT, env=WORKER_ENV,
                                   stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
                                   start_new_session=True)
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
                stream.write(line)
                stream.flush()
            return_code = process.wait()
        except BaseException:
            import signal
            os.killpg(process.pid, signal.SIGTERM)
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                os.killpg(process.pid, signal.SIGKILL)
            raise
    if return_code:
        raise RuntimeError(f"Worker failed ({return_code}); see {log}")


## 5. Observed endpoints and canonical fitting

Endpoint preparation validates shape, finite values, range and agreement of the endpoint affine matrices. Source hashes are recorded for later verification. The intermediate scans are not read in this section.


In [ ]:
for case in TEST_CASES:
    run_worker("prepare", case)
print(f"Prepared {len(TEST_CASES)} cases with phases 0% and 50% only.")


### Endpoint quality control

The first, middle and last test series and slice 64 are selected before evaluation. Both observed endpoints use the same grayscale range.


In [ ]:
def save_figure(fig, name):
    for extension in ["png", "pdf"]:
        fig.savefig(ROOT / "figures" / f"{name}.{extension}", dpi=300,
                    bbox_inches="tight", facecolor="white")
    plt.show()
    plt.close(fig)


fig, axes = plt.subplots(len(QUALITATIVE_CASES), 2, figsize=(6, 8), constrained_layout=True)
for row, case in enumerate(QUALITATIVE_CASES):
    for col, phase in enumerate([0, 50]):
        volume = np.load(ROOT / "prepared" / case / f"phase_{phase:02d}.npy", mmap_mode="r")
        axes[row, col].imshow(volume[QUALITATIVE_SLICE], cmap="gray", vmin=0, vmax=1)
        axes[row, col].set_title(f"{case} · {phase}%")
        axes[row, col].axis("off")
save_figure(fig, "observed_endpoints")


In [ ]:
for case in TEST_CASES:
    run_worker("canonical", case)
print("All canonical models are ready.")


## 6. Temporal fitting

Each method starts from a newly initialized temporal network and the same frozen canonical model. The 50% endpoint is the only temporal image target in the baseline and pseudo variants; auxiliary reconstruction in cycle variants also uses 0%. Model choice is the configured final iteration, with no hidden-phase validation.


In [ ]:
for case in TEST_CASES:
    for method in METHODS:
        run_worker("train", case, method)
print(f"Completed temporal fitting: {len(METHODS)} methods × {len(TEST_CASES)} series.")


### Training time and learning curves

Training curves describe reconstruction of the observed 50% endpoint. All methods are evaluated at the final iteration.


In [ ]:
def temporal_dir(case, method):
    return ROOT / "methods" / f"benchmark_{case}" / method


records = []
for case in TEST_CASES:
    for method in METHODS:
        done = json.loads((temporal_dir(case, method) / "completion.json").read_text())
        assert done["iteration"] == TEMPORAL_ITERATIONS
        records.append({"Case": case, "Method": method,
                        "TrainingSeconds": done["elapsed_seconds"],
                        "PrimaryParameters": done["parameter_count"]})
training_summary = pd.DataFrame(records)
training_summary.to_csv(ROOT / "tables/training_summary.csv", index=False)
display(training_summary.groupby("Method", sort=False)[
    ["TrainingSeconds", "PrimaryParameters"]].mean().reindex(METHODS))


In [ ]:
print(f"Observed 50% endpoint training curves: mean L1 and PSNR across {len(TEST_CASES)} selected series:")
fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
for method in METHODS:
    histories = []
    for case in TEST_CASES:
        h = pd.read_csv(temporal_dir(case, method) / "training_history.csv")
        histories.append(h.set_index("Iteration")[["L1", "PSNR"]])
    mean_history = pd.concat(histories).groupby(level=0).mean().rolling(100, min_periods=1).mean()
    for axis, metric in zip(axes, ["L1", "PSNR"]):
        axis.plot(mean_history.index, mean_history[metric], label=LABELS[method], linewidth=1)
        axis.set(xlabel="Iteration", ylabel=f"Observed 50% {metric}")
axes[1].legend(fontsize=8)
save_figure(fig, "endpoint_training_curves")


## 7. Predictions

The final primary fields render 336 hidden-phase volumes: 14 series × 6 methods × 4 phases. Predictions are saved in `(Z,Y,X)` order. Intermediate ground truth is read in the next section, for evaluation.


In [ ]:
print(f"Rendering {len(TEST_CASES) * len(METHODS) * len(PHASES)} volumes from the final checkpoints:")
for case in TEST_CASES:
    for method in METHODS:
        run_worker("render", case, method)
print(f"All {len(TEST_CASES) * len(METHODS) * len(PHASES)} predictions are ready.")


## 8. Full-volume evaluation

The models are now frozen. Each predicted volume is compared with its matching CT volume using the UVI-Net metric definitions:
\[
\operatorname{PSNR}=-10\log_{10}(\operatorname{MSE}),\qquad
\operatorname{NMSE}[\%]=100\frac{\operatorname{MSE}}{\operatorname{mean}(I^2)}.
\]
NCC is the mean squared local correlation with a $9^3$ box window and denominator stabilizer $10^{-5}$. SSIM uses an $11^3$ Gaussian window, $\sigma=1.5$, $C_1=0.01^2$, $C_2=0.03^2$. Both use zero padding. These are **volume metrics**, not averages of slice metrics. There is no ROI mask or per-volume intensity rescaling.

The main table averages 56 case–phase measurements per method and follows the [official AverageMeter](https://github.com/jungeun122333/UVI-Net/blob/main/utils/utils.py): population standard deviation divided by $\sqrt{56}$ (`ddof=0`). Sample-SD standard error (`ddof=1`) is also exported explicitly. Case–phase entries are correlated within patients: these descriptive errors are not patient-level confidence intervals. Per-case, per-phase and per-patient tables are retained.


In [ ]:
print("Evaluation setup: original full-volume 3D SSIM and NCC definitions.")
from functools import partial
METRIC_DEVICE = torch.device(f"cuda:{GPU_INDEX}")
torch.cuda.set_device(METRIC_DEVICE)
ssim3d, ncc3d = benchmark_metrics.ssim3d, benchmark_metrics.ncc3d
evaluate_volume = partial(benchmark_metrics.evaluate_volume, device=METRIC_DEVICE)


In [ ]:
# This is the first section that reads intermediate CT ground truth.
print(f"Evaluating {len(TEST_CASES) * len(METHODS) * len(PHASES)} volumes: PSNR, NCC, SSIM and NMSE (%).")
for case in TEST_CASES:
    for method in METHODS:
        assert (ROOT / "predictions" / case / method / "render.json").is_file()

rows, ground_truth_sources = [], []
for case in TEST_CASES:
    reference = json.loads((ROOT / "prepared" / case / "observed_sources.json").read_text())
    for phase in PHASES:
        path = DATA_ROOT / case / f"ct_{case}_frame{phase // 10}.nii.gz"
        image = nib.load(str(path))
        assert np.allclose(image.affine, reference["headers"]["0"]["affine"])
        gt = image.get_fdata(dtype=np.float32).transpose(2, 1, 0).copy()
        assert np.isfinite(gt).all() and gt.min() >= -1e-6 and gt.max() <= 1 + 1e-6
        ground_truth_sources.append({"Case": case, "Phase": phase, "Path": str(path), "SHA256": sha256_file(path)})
        for method in METHODS:
            directory = ROOT / "predictions" / case / method
            prediction_path = directory / f"phase_{phase:02d}.npy"
            render = json.loads((directory / "render.json").read_text())
            assert sha256_file(prediction_path) == render["prediction_sha256"][str(phase)]
            score = evaluate_volume(gt, np.load(prediction_path))
            rows.append({"Case": case, "Patient": case.split("_")[0], "Phase": phase, "Method": method, **score})
        pd.DataFrame(rows).to_csv(ROOT / "metrics/per_case_phase.csv", index=False)
    print("Evaluated:", case)
metrics = pd.DataFrame(rows)
assert len(metrics) == len(TEST_CASES) * len(METHODS) * len(PHASES)
assert not metrics.duplicated(["Case", "Phase", "Method"]).any()
assert metrics.groupby("Method").size().eq(len(TEST_CASES) * len(PHASES)).all()
write_json(ROOT / "metrics/ground_truth_sources.json", {"files": ground_truth_sources})
print("Evaluation complete:", len(metrics), "volumes.")


## 9. Benchmark tables

The main table preserves all official test series. The exclusion of `118_0` is supplementary. No UVI-Net score is inserted automatically: external published values should be labeled with their source and setting, separately from results generated by this run.


In [ ]:
METRICS = ["PSNR", "NCC", "SSIM", "NMSE"]


def summary_table(frame):
    rows = []
    for method in METHODS:
        group = frame.loc[frame.Method == method]
        row = {"Method": method, "N": len(group)}
        for metric in METRICS:
            values = group[metric].to_numpy()
            row[metric + "_mean"] = float(values.mean())
            row[metric + "_SE"] = float(values.std(ddof=0) / np.sqrt(len(values)))
            row[metric + "_SE_ddof1"] = float(values.std(ddof=1) / np.sqrt(len(values)))
        rows.append(row)
    return pd.DataFrame(rows)


summary = summary_table(metrics)
summary.to_csv(ROOT / "tables/overall_numeric.csv", index=False)
formatted = pd.DataFrame({"Method": [LABELS[m] for m in METHODS]})
for metric, title in zip(METRICS, ["PSNR ↑", "NCC ↑", "SSIM ↑", "NMSE (%) ↓"]):
    formatted[title] = [f"{r[metric + '_mean']:.3f} ± {r[metric + '_SE']:.3f}" for r in summary.to_dict("records")]
formatted.to_csv(ROOT / "tables/main_table.csv", index=False)
(ROOT / "tables/main_table.tex").write_text(formatted.to_latex(index=False, escape=True))

display(formatted)
summary_table(metrics.loc[metrics.Case != "118_0"]).to_csv(ROOT / "tables/sensitivity_without_118_0.csv", index=False)
for keys, name in [(["Method", "Case"], "per_case"), (["Method", "Phase"], "per_phase"),
                   (["Method", "Patient"], "per_patient")]:
    metrics.groupby(keys, sort=False)[METRICS].mean().reset_index().to_csv(ROOT / "tables" / f"{name}.csv", index=False)

# Paired descriptive differences on exactly the same cases and phases.
base = metrics.loc[metrics.Method == "baseline"].set_index(["Case", "Phase"])[METRICS]
differences = []
for method in METHODS[1:]:
    other = metrics.loc[metrics.Method == method].set_index(["Case", "Phase"])[METRICS]
    delta = other.subtract(base).reset_index()
    delta["Method"] = method
    differences.append(delta)
pd.concat(differences).to_csv(ROOT / "tables/paired_differences_from_baseline.csv", index=False)
print("Tables saved:", ROOT / "tables")


In [ ]:
print("Rendering and saving the publication-ready summary table for the six reconstruction methods.")
fig, ax = plt.subplots(figsize=(11, 3.3), constrained_layout=True)
fig.patch.set_facecolor("white")
ax.set_facecolor("white")
ax.axis("off")
table = ax.table(cellText=formatted.values, colLabels=formatted.columns, loc="center", cellLoc="center")
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 1.8)
for (row, col), cell in table.get_celld().items():
    cell.set_text_props(color="black")
    if row == 0:
        cell.set_facecolor("#dce6f1")
        cell.set_text_props(weight="bold")
    elif row % 2 == 0:
        cell.set_facecolor("#f4f6f8")
ax.set_title(
    f"4D-Lung · official 14-series split · mean ± SE over {len(TEST_CASES) * len(PHASES)} volumes",
    pad=16,
    color="black",
)
save_figure(fig, "main_table")


### Metrics by respiratory phase

Each point is the mean over the same 14 series. Error bars show sample standard error over those series; they are descriptive because several series share a patient.


In [ ]:
print(f"Plotting phase-wise quality across {len(TEST_CASES)} selected series.")
fig, axes = plt.subplots(2, 2, figsize=(11, 8), constrained_layout=True)
for axis, metric in zip(axes.flat, METRICS):
    for method in METHODS:
        group = metrics.loc[metrics.Method == method].groupby("Phase")[metric]
        axis.errorbar(PHASES, group.mean().reindex(PHASES), yerr=group.sem().reindex(PHASES),
                      marker="o", markersize=3, capsize=2, linewidth=1, label=LABELS[method])
    axis.set(xlabel="Respiratory phase (%)", ylabel=metric + (" (%) ↓" if metric == "NMSE" else " ↑"), xticks=PHASES)
axes[0, 0].legend(fontsize=8)
save_figure(fig, "metrics_by_phase")


## 10. Qualitative comparison

The cases and slice were fixed before evaluation. Images share `[0,1]`; absolute errors share `[0,0.2]`. Figures show all methods and all four hidden phases, with the same ordering and scale. They are not selected from the best-scoring results.


In [ ]:
print("Generating qualitative reconstruction grids and absolute-error maps for the selected held-out cases.")
for case in QUALITATIVE_CASES:
    gt = {}
    for phase in PHASES:
        path = DATA_ROOT / case / f"ct_{case}_frame{phase // 10}.nii.gz"
        gt[phase] = nib.load(str(path)).get_fdata(dtype=np.float32).transpose(2, 1, 0)[QUALITATIVE_SLICE]
    fig, axes = plt.subplots(7, 4, figsize=(10, 15), constrained_layout=True)
    for col, phase in enumerate(PHASES):
        axes[0, col].imshow(gt[phase], cmap="gray", vmin=0, vmax=1)
        axes[0, col].set_title(f"{phase}%")
        for row, method in enumerate(METHODS, start=1):
            p = np.load(ROOT / "predictions" / case / method / f"phase_{phase:02d}.npy", mmap_mode="r")
            axes[row, col].imshow(p[QUALITATIVE_SLICE], cmap="gray", vmin=0, vmax=1)
    for row, label in enumerate(["Ground truth"] + [LABELS[m] for m in METHODS]):
        axes[row, 0].set_ylabel(label, fontsize=9)
    for ax in axes.flat:
        ax.set_xticks([]);
        ax.set_yticks([])
    fig.suptitle(f"{case} · axial slice {QUALITATIVE_SLICE}")
    save_figure(fig, f"reconstruction_{case}")

    fig, axes = plt.subplots(6, 4, figsize=(10, 13), constrained_layout=True)
    for row, method in enumerate(METHODS):
        for col, phase in enumerate(PHASES):
            p = np.load(ROOT / "predictions" / case / method / f"phase_{phase:02d}.npy", mmap_mode="r")
            im = axes[row, col].imshow(np.abs(p[QUALITATIVE_SLICE] - gt[phase]), cmap="magma", vmin=0, vmax=ERROR_VMAX)
            if row == 0:
                axes[row, col].set_title(f"{phase}%")
        axes[row, 0].set_ylabel(LABELS[method], fontsize=9)
    for ax in axes.flat:
        ax.set_xticks([]);
        ax.set_yticks([])
    fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.5, label="Absolute intensity error")
    fig.suptitle(f"{case} · absolute error · slice {QUALITATIVE_SLICE}")
    save_figure(fig, f"absolute_error_{case}")


In [ ]:
print("Generating full-range absolute-error maps on the fixed [0, 1] intensity scale for qualitative diagnostics.")

for case in QUALITATIVE_CASES:
    gt = {}
    for phase in PHASES:
        path = DATA_ROOT / case / f"ct_{case}_frame{phase // 10}.nii.gz"
        gt[phase] = (
            nib.load(str(path))
            .get_fdata(dtype=np.float32)
            .transpose(2, 1, 0)[QUALITATIVE_SLICE]
        )

    fig, axes = plt.subplots(6, 4, figsize=(10, 13), constrained_layout=True)

    for row, method in enumerate(METHODS):
        for col, phase in enumerate(PHASES):
            p = np.load(
                ROOT / "predictions" / case / method / f"phase_{phase:02d}.npy",
                mmap_mode="r",
            )

            im = axes[row, col].imshow(
                np.abs(p[QUALITATIVE_SLICE] - gt[phase]),
                cmap="magma",
                vmin=0,
                vmax=1.0,
            )

            if row == 0:
                axes[row, col].set_title(f"{phase}%")

        axes[row, 0].set_ylabel(LABELS[method], fontsize=9)

    for ax in axes.flat:
        ax.set_xticks([])
        ax.set_yticks([])

    fig.colorbar(
        im,
        ax=axes.ravel().tolist(),
        shrink=0.5,
        label="Absolute intensity error",
    )

    fig.suptitle(
        f"{case} · absolute error · full [0, 1] scale · slice {QUALITATIVE_SLICE}"
    )

    save_figure(fig, f"absolute_error_full_range_{case}")

## 11. Computation and export

Temporal training time includes the additional auxiliary work; equal iteration budgets are therefore not equal time budgets. Inference uses only the primary field. The canonical representation is shared across methods, so its fitting cost is listed separately rather than counted six times. Allocator peaks are process-local measurements, not total device utilization.


In [ ]:
print("Summarizing training and rendering cost, GPU memory, parameter count, and clipping diagnostics across the selected series.")
resource_rows = []
for case in TEST_CASES:
    for method in METHODS:
        run = temporal_dir(case, method)
        history = pd.read_csv(run / "training_history.csv")
        rendered = json.loads((ROOT / "predictions" / case / method / "render.json").read_text())
        audit = training_summary.loc[(training_summary.Case == case) & (training_summary.Method == method)].iloc[0]
        resource_rows.append({"Case": case, "Method": method,
                              "TrainingSeconds": audit.TrainingSeconds,
                              "TrainingPeakGpuGB": history.PeakGpuMemoryGB.max(),
                              "RenderSecondsFourVolumes": rendered["seconds"],
                              "RenderPeakGpuGB": rendered["peak_gpu_gb"],
                              "PrimaryParameters": rendered["primary_parameters"],
                              "MaxClippedRGBFraction": max(rendered["rgb_clipped_fraction"].values()),
                              })
resources = pd.DataFrame(resource_rows)
resources.to_csv(ROOT / "tables/resources_per_case.csv", index=False)
resource_summary = resources.groupby("Method").mean(numeric_only=True).reindex(METHODS)
resource_summary.to_csv(ROOT / "tables/resources_mean.csv")
display(resource_summary)
canonical_cost = []
for case in TEST_CASES:
    folder = ROOT / "canonical" / f"benchmark_{case}" / f"phase00_iter{CANONICAL_ITERATIONS}"
    metadata = folder / "resource_segment.json"
    if metadata.exists():
        canonical_cost.append({"Case": case, **json.loads(metadata.read_text())})
pd.DataFrame(canonical_cost).to_csv(ROOT / "tables/canonical_resources.csv", index=False)


In [ ]:
print("Plotting the mean training and inference cost of all six reconstruction methods across the selected series.")
fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
labels = [LABELS[m] for m in METHODS]
axes[0].bar(labels, resource_summary.TrainingSeconds / 60)
axes[0].set_ylabel("Mean temporal fitting time (min)")
axes[1].bar(labels, resource_summary.RenderSecondsFourVolumes / 4)
axes[1].set_ylabel("Mean inference time per volume (s)")
for ax in axes:
    ax.tick_params(axis="x", labelrotation=35)
save_figure(fig, "computation_cost")


In [ ]:
print("Saving tables, figures, metrics and experiment settings.")
# A compact result bundle excludes large checkpoints and prediction arrays.
import zipfile

bundle = ROOT / "benchmark_results.zip"
with zipfile.ZipFile(bundle, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for name in ["config", "metrics", "tables", "figures", "environment"]:
        for path in sorted((ROOT / name).rglob("*")):
            if path.is_file() and "__pycache__" not in path.parts:
                archive.write(path, path.relative_to(ROOT))
write_json(ROOT / "benchmark_complete.json", {
    "utc": datetime.now(timezone.utc).isoformat(), "experiment": EXPERIMENT_ID,
    "canonical_runs": len(TEST_CASES), "temporal_runs": len(TEST_CASES) * len(METHODS), "evaluated_volumes": len(metrics),
    "result_bundle_sha256": sha256_file(bundle),
})
print("Results:", bundle)
print("Checkpoints:", ROOT / "canonical", "and", ROOT / "methods")
print("Predictions:", ROOT / "predictions")
from IPython.display import FileLink

# Jupyter links are relative to the notebook working directory.
display(FileLink(os.path.relpath(bundle, Path.cwd())))


## Interpretation

Report the six variants together, including negative results. A gain for `trajectory` supports its trajectory prior; a gain for `consistent_cycle` supports the added consistency objective; `trajectory_cycle` tests their combination. Geometry consistency can still describe an incorrect motion, so endpoint reconstruction alone cannot establish interpolation quality.

Use the official 14-series table for comparison with UVI-Net and show the setting as instance-specific endpoint fitting. Accompany it with computation cost, per-patient results, the supplementary exclusion of `118_0`, and the fixed seed and iteration budget. Generalization beyond two test patients or stability across seeds requires additional experiments.
